# Checkpoint A: Tái hiện quy trình Demo của Giảng viên

Jupyter Notebook này giúp học viên tự động chạy lại toàn bộ quy trình thị phạm của giảng viên ở **Phần A: Demo dẫn nhập của giảng viên** trong [lab.md](../../lab.md).

### Các công việc notebook này thực hiện:
1. **Đọc dữ liệu NOC log thô** từ tệp giả lập `synthetic-data/sample-noc-alerts.csv`.
2. **Làm sạch PII (Thông tin định danh cá nhân):** Che giấu họ tên, số điện thoại, email của kỹ sư trực ca xuất hiện trong log bằng nhãn `[REDACTED_PII]`.
3. **Thống kê phân loại theo loại thiết bị (Device Type):** Đếm số lượng cảnh báo cho từng loại thiết bị và sinh mã biểu đồ Mermaid pie chart tương ứng.
4. **Lọc cảnh báo nguy kịch (Critical) đang mở (Open):** Xác định chính xác 3 cảnh báo nguy kịch đang mở, dịch lỗi sang tiếng Việt và đề xuất quy trình ứng cứu nhanh HITL.
5. **Xuất báo cáo Markdown** ra tệp `outputs/noc-alert-sanitized-report.md`.

---

### Bước A1: Đọc dữ liệu log NOC thô

In [1]:
import os
import re
import csv

# Thiết lập đường dẫn tương đối
CSV_PATH = "../../synthetic-data/sample-noc-alerts.csv"
OUTPUT_DIR = "../../outputs"
REPORT_PATH = os.path.join(OUTPUT_DIR, "noc-alert-sanitized-report.md")

print("=== Bước A1: Đọc log NOC thô ===")
alerts = []
if os.path.exists(CSV_PATH):
    with open(CSV_PATH, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            alerts.append(dict(row))
    print(f"[OK] Đọc thành công {len(alerts)} dòng cảnh báo thô từ {CSV_PATH}")
    print("Hiển thị 3 dòng dữ liệu đầu tiên:")
    for i in range(min(3, len(alerts))):
        print(f"  - {alerts[i]}")
else:
    print(f"[FAIL] Không tìm thấy file dữ liệu tại {CSV_PATH}")

=== Bước A1: Đọc log NOC thô ===
[OK] Đọc thành công 115 dòng cảnh báo thô từ ../../synthetic-data/sample-noc-alerts.csv
Hiển thị 3 dòng dữ liệu đầu tiên:
  - {'alert_id': 'ALERT-001', 'timestamp': '2026-05-01 08:02:00', 'site_code': 'TEST_SITE_018', 'device_type': 'server', 'severity': 'high', 'summary': 'database query latency spike in performance test [Contact: Nguyen Van A (0987654321)]', 'status': 'open'}
  - {'alert_id': 'ALERT-002', 'timestamp': '2026-05-01 21:16:00', 'site_code': 'TEST_SITE_009', 'device_type': 'switch', 'severity': 'low', 'summary': 'interface flap in synthetic log', 'status': 'open'}
  - {'alert_id': 'ALERT-003', 'timestamp': '2026-05-01 23:59:00', 'site_code': 'TEST_SITE_026', 'device_type': 'server', 'severity': 'high', 'summary': 'unauthorized login attempt simulated in audit log', 'status': 'resolved'}


### Bước A2: Làm sạch dữ liệu nhạy cảm (PII)

Chúng ta sử dụng Regex để phát hiện và thay thế các thông tin cá nhân nhạy cảm trong cột `summary`:
- Số điện thoại (10 chữ số bắt đầu bằng số 0)
- Địa chỉ Email (chứa ký tự `@` và tên miền)
- Họ tên kỹ sư trực ca sau các từ khóa chỉ định (như `Contact: `, `Assigned to: `, `engineer `, `owner: `)

In [2]:
def sanitize_pii(text):
    if not isinstance(text, str):
        return text
    
    # 1. Che giấu email
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[REDACTED_PII]', text)
    
    # 2. Che giấu số điện thoại
    text = re.sub(r'\b0\d{9}\b', '[REDACTED_PII]', text)
    
    # 3. Che giấu họ tên kỹ sư sau các từ khóa chỉ định
    text = re.sub(r'(Contact:\s+|Assigned to:\s+|engineer\s+|owner:\s+|engineer at\s+)([A-ZÀ-Ỹ][a-zà-ỹ]+(\s+[A-ZÀ-Ỹ][a-zà-ỹ]+)+)', r'\1[REDACTED_PII]', text)
    
    return text

print("=== Bước A2: Làm sạch dữ liệu PII ===")
for alert in alerts:
    alert['summary'] = sanitize_pii(alert['summary'])

print("Hiển thị một số ví dụ log sau khi được làm sạch PII:")
example_count = 0
for alert in alerts:
    if "[REDACTED_PII]" in alert['summary']:
        print(f"  - ID: {alert['alert_id']} | Summary: {alert['summary']}")
        example_count += 1
        if example_count >= 5:
            break

=== Bước A2: Làm sạch dữ liệu PII ===
Hiển thị một số ví dụ log sau khi được làm sạch PII:
  - ID: ALERT-001 | Summary: database query latency spike in performance test [Contact: [REDACTED_PII] A ([REDACTED_PII])]
  - ID: ALERT-009 | Summary: RF module communication failure in lab environment [Assigned to: [REDACTED_PII] E ([REDACTED_PII])]
  - ID: ALERT-013 | Summary: high memory usage alert on training cluster [Contact: [REDACTED_PII] A ([REDACTED_PII])]
  - ID: ALERT-022 | Summary: database query latency spike in performance test [Contact: [REDACTED_PII] B ([REDACTED_PII])]
  - ID: ALERT-025 | Summary: interface flap in synthetic log [Contact: [REDACTED_PII] B ([REDACTED_PII])]


### Bước A3: Thống kê phân loại theo loại thiết bị (Device Type)

Chúng ta đếm tần suất xuất hiện cảnh báo của mỗi loại thiết bị (`device_type`) và biểu diễn chúng dưới dạng bảng thống kê Markdown cùng biểu đồ Mermaid.

In [3]:
print("=== Bước A3: Thống kê theo loại thiết bị ===")
device_counts = {}
for alert in alerts:
    dev = alert['device_type']
    device_counts[dev] = device_counts.get(dev, 0) + 1

# Sắp xếp thống kê giảm dần theo số lượng
sorted_devices = sorted(device_counts.items(), key=lambda x: x[1], reverse=True)

# Tạo bảng Markdown thống kê
device_table_md = "| Loại thiết bị (Device Type) | Số lượng cảnh báo |\n| --- | ---: |\n"
for device, count in sorted_devices:
    device_table_md += f"| {device.capitalize()} | {count} |\n"

# Tạo biểu đồ Mermaid pie chart
mermaid_pie_md = "```mermaid\npie title Phân loại cảnh báo theo loại thiết bị\n"
for device, count in sorted_devices:
    mermaid_pie_md += f'    "{device.capitalize()}" : {count}\n'
mermaid_pie_md += "```"

print("\n--- Bảng thống kê Markdown ---")
print(device_table_md)
print("--- Mã biểu đồ Mermaid ---")
print(mermaid_pie_md)

=== Bước A3: Thống kê theo loại thiết bị ===

--- Bảng thống kê Markdown ---
| Loại thiết bị (Device Type) | Số lượng cảnh báo |
| --- | ---: |
| Server | 23 |
| Ups | 22 |
| Base-station | 19 |
| Switch | 18 |
| Firewall | 18 |
| Router | 15 |

--- Mã biểu đồ Mermaid ---
```mermaid
pie title Phân loại cảnh báo theo loại thiết bị
    "Server" : 23
    "Ups" : 22
    "Base-station" : 19
    "Switch" : 18
    "Firewall" : 18
    "Router" : 15
```


### Bước A4: Trích xuất các cảnh báo nguy kịch (Critical) đang mở (Open)

Chúng ta lọc ra đúng 03 cảnh báo nguy kịch (`severity == 'critical'`) và đang ở trạng thái mở (`status == 'open'`).
Sau đó dịch tóm tắt lỗi sang tiếng Việt và đề xuất quy trình ứng cứu nhanh có sự tham gia của con người trong vòng lặp (Human-in-the-loop - HITL).

In [4]:
print("=== Bước A4: Lọc cảnh báo Critical & Open ===")
critical_open_alerts = [a for a in alerts if a['severity'].lower() == 'critical' and a['status'].lower() == 'open']
print(f"Tìm thấy {len(critical_open_alerts)} cảnh báo nguy kịch đang mở:")
for a in critical_open_alerts:
    print(f"  - ID: {a['alert_id']} | Site: {a['site_code']} | Device: {a['device_type']} | Summary: {a['summary']}")

# Định nghĩa từ điển dịch và quy trình ứng cứu HITL cho 3 lỗi cụ thể
hitl_procedures = {
    "ALERT-040": {
        "dich_tieng_viet": "Cảnh báo nguồn dự phòng trong kịch bản đào tạo",
        "hitl_flow": "Kỹ sư trực NOC L2 liên hệ khẩn cấp với đội kỹ thuật tại trạm TEST_SITE_034 để kiểm tra trạng thái hoạt động của máy phát điện hoặc ắc quy dự phòng tại chỗ."
    },
    "ALERT-058": {
        "dich_tieng_viet": "Lỗi kết nối mô-đun RF trong môi trường phòng lab",
        "hitl_flow": "Kỹ sư NOC L2 thực hiện kiểm tra ping/kết nối quang đến mô-đun RF của trạm TEST_SITE_013. Nếu không phục hồi, điều phối kỹ thuật viên onsite L3 đến phòng lab để thay thế nóng mô-đun lỗi."
    },
    "ALERT-081": {
        "dich_tieng_viet": "Phát hiện lưu lượng quảng bá (broadcast traffic) vượt ngưỡng nguy kịch trong mạng con phòng lab",
        "hitl_flow": "Kỹ sư NOC L2 truy cập Switch tại TEST_SITE_020, xác định cổng (port) phát sinh lưu lượng lớn, thực hiện giới hạn băng thông quảng bá (storm control) hoặc cô lập VLAN bị ảnh hưởng tạm thời để tránh nghẽn toàn hệ thống mạng."
    }
}

# Tạo báo cáo chi tiết cho 3 lỗi
critical_report_md = ""
for row in critical_open_alerts:
    aid = row['alert_id']
    info = hitl_procedures.get(aid, {
        "dich_tieng_viet": "Cảnh báo chưa được dịch",
        "hitl_flow": "Cần con người kiểm tra trực tiếp"
    })
    
    critical_report_md += f"#### {aid} - Trạm {row['site_code']} ({row['device_type'].capitalize()})\n"
    critical_report_md += f"- **Thời gian xảy ra:** `{row['timestamp']}`\n"
    critical_report_md += f"- **Mô tả lỗi gốc:** *{row['summary']}*\n"
    critical_report_md += f"- **Dịch tiếng Việt:** **{info['dich_tieng_viet']}**\n"
    critical_report_md += f"- **Quy trình ứng cứu nhanh (HITL):** {info['hitl_flow']}\n\n"

print("\n--- Nội dung báo cáo cảnh báo nguy kịch ---")
print(critical_report_md)

=== Bước A4: Lọc cảnh báo Critical & Open ===
Tìm thấy 3 cảnh báo nguy kịch đang mở:
  - ID: ALERT-040 | Site: TEST_SITE_034 | Device: base-station | Summary: power backup warning in training scenario
  - ID: ALERT-058 | Site: TEST_SITE_013 | Device: base-station | Summary: RF module communication failure in lab environment
  - ID: ALERT-081 | Site: TEST_SITE_020 | Device: switch | Summary: high broadcast traffic detected in lab subnet

--- Nội dung báo cáo cảnh báo nguy kịch ---
#### ALERT-040 - Trạm TEST_SITE_034 (Base-station)
- **Thời gian xảy ra:** `2026-05-01 23:56:00`
- **Mô tả lỗi gốc:** *power backup warning in training scenario*
- **Dịch tiếng Việt:** **Cảnh báo nguồn dự phòng trong kịch bản đào tạo**
- **Quy trình ứng cứu nhanh (HITL):** Kỹ sư trực NOC L2 liên hệ khẩn cấp với đội kỹ thuật tại trạm TEST_SITE_034 để kiểm tra trạng thái hoạt động của máy phát điện hoặc ắc quy dự phòng tại chỗ.

#### ALERT-058 - Trạm TEST_SITE_013 (Base-station)
- **Thời gian xảy ra:** `2026-05-

### Bước A5: Kết xuất báo cáo Markdown hoàn chỉnh

Tổng hợp toàn bộ kết quả trên và xuất ra tệp `outputs/noc-alert-sanitized-report.md` để kết thúc quy trình thị phạm.

In [5]:
print("=== Bước A5: Kết xuất báo cáo Markdown ===")

# Đảm bảo thư mục outputs tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)

report_content = f"""---
mo-ta: bao cao phan tich log NOC da lam sach du lieu nhay cam PII va loc loi nguy kich
trang-thai: active
phien-ban: v1.0
created-at: 2026-06-11 08:58 +07:00
updated-at: 2026-06-11 08:58 +07:00
---

# Báo cáo phân tích và làm sạch log NOC tuần tự động

Báo cáo này được tự động tạo bởi Checkpoint A nhằm tái hiện và kiểm tra kết quả thị phạm phân tích dữ liệu log NOC.

## 1. Kết quả làm sạch thông tin định danh cá nhân (PII)
Toàn bộ thông tin cá nhân của các kỹ sư trực ca (Họ tên, Số điện thoại, Email) xuất hiện trong cột chi tiết log thô đã được phát hiện và che giấu bằng nhãn `[REDACTED_PII]` để đảm bảo an toàn thông tin dữ liệu.

## 2. Thống kê số lượng cảnh báo theo loại thiết bị (Device Type)

{device_table_md}
### Biểu đồ phân loại trực quan (Mermaid Pie Chart)

{mermaid_pie_md}

## 3. Danh sách cảnh báo nguy kịch (Critical) đang mở (Open) & Quy trình ứng cứu (HITL)

Dưới đây là 3 cảnh báo nguy kịch đang mở cần được ưu tiên xử lý lập tức kèm quy trình ứng cứu nhanh có con người tham gia kiểm duyệt (Human-in-the-loop):

{critical_report_md}
"""

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report_content.strip())

print(f"[OK] Đã xuất báo cáo thành công tại: {REPORT_PATH}")
print("Học viên có thể mở tệp này trên IDE để kiểm tra nội dung và biểu đồ trực quan.")

=== Bước A5: Kết xuất báo cáo Markdown ===
[OK] Đã xuất báo cáo thành công tại: ../../outputs/noc-alert-sanitized-report.md
Học viên có thể mở tệp này trên IDE để kiểm tra nội dung và biểu đồ trực quan.
